# PQC Across Your Infrastructure

Each cell runs one command you would type,
shows the real output, and the last cell of each beat draws a compact
**two-entity panel** (who talks to who + expected-vs-measured evidence).

**Entity handles** run a command inside a container and echo it under that
entity's name, e.g. `authenticator("hostapd -dd /cfg/hostapd.conf")` runs
`hostapd` inside the authenticator container. `host("…")` runs on your host. A
trailing `&` backgrounds a command so it survives to the next cell.

**How to drive it**
1. Run **Setup** once (build is one-time per machine).
2. Walk each demo top-to-bottom, **one cell per action**. Read the comment,
   run the cell, read the real output, then move to the next cell. The
   `panel(...)` cell at the end of each beat is the visual to follow.
3. `panic_reset()` clears leftover and stuck processes; `cleanup_all()` restores defaults.

<a id="toc"></a>
## Contents

Run **Setup** once and then jump straight into any demo.

- [Setup](#setup)
- **IPsec / IKEv2**
  - [Key Exchange](#kex-ipsec)
  - [Authentication](#auth-ipsec)
- **TLS 1.3**
  - [Key Exchange](#kex-tls)
  - [Authentication](#auth-tls)
- **MACsec / 802.1X**
  - [Key Exchange & Authentication](#macsec)
- **SSH**
  - [Key Exchange & Authentication](#ssh)
- [Cleanup](#wrapup)

<a id="setup"></a>
## Setup

Imports the entity handles below, then builds + starts **every** lab.
`preflight_build()` is **one-time** per machine; then go straight to
`preflight_up()`.

In [ ]:
import sys, pathlib
# Make pqc_demo.py importable (it sits next to this notebook in demos/containers_demo/).
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'containers_demo', pathlib.Path.cwd() / 'demos' / 'containers_demo'):
    if (_p / 'pqc_demo.py').exists():
        sys.path.insert(0, str(_p)); break
from pqc_demo import *  # noqa: F401,F403

# Ladders for the end-of-beat panels (the commands run via the entity handles).
ipsec = IPsecDemo(); ipsec_auth = IPsecAuthDemo()
tls = TLSDemo(); tls_auth = TLSAuthDemo()
macsec = MACsecDemo(); ssh = SSHDemo()

set_presenter_mode(True)
print('Working directory:', REPO_ROOT)
print('Entities you drive:')
print('  IPsec:  initiator responder | auth_initiator auth_responder')
print('  TLS:    tls_server tls_client | mtls_server mtls_client')
print('  MACsec: authenticator supplicant')
print('  SSH:    ssh_server ssh_client')
print('  Host:   host')

In [ ]:
# One-time per machine; compiles every lab image from source (slow).
preflight_build()

In [ ]:
# Bring every stack up so the on-stage cost is only the handshake.
preflight_up()

In [ ]:
preflight_status()

---
# IPsec / IKEv2 for Layer 3 VPNs

<a id="kex-ipsec"></a>
## IKEv2 Key Exchange

IKEv2 carries ML-KEM in an extra **IKE_INTERMEDIATE**
round trip (RFC 9370), and the ML-KEM public key is large enough to force
**IKE-layer fragmentation**.

**Beat 1: bring up the hybrid tunnel (watch the live IKE exchange)**

In [ ]:
# Initiator: the swanctl.conf knob that adds ML-KEM on top of the classical DH group
#   proposals ending in '-ke1_mlkem768' is what makes this hybrid (the commented-out
#   line below it is the classical-only alternative); fragmentation is required
#   because the ML-KEM public key is too big to fit in one unfragmented IKE message.
initiator("grep -E -A1 'proposals|fragmentation' /usr/local/etc/swanctl/swanctl.conf");

In [ ]:
# Initiator: make sure no previous tunnel/capture is still running
initiator("swanctl --terminate --ike pqc-tunnel 2>/dev/null || true; true")
initiator("pkill -f [t]cpdump 2>/dev/null; true");

In [ ]:
# Initiator: capture IKE (UDP 500/4500), then initiate the hybrid child SA
initiator("tcpdump -i any --immediate-mode -U -n 'port 500 or port 4500' -w /tmp/hybrid.pcap &")
initiator("swanctl --initiate --child pqc-child");

In [ ]:
# Initiator: read the negotiated IKE SA suite off swanctl
suite = initiator("swanctl --list-sas | grep -iE 'ML_KEM|CURVE_25519' || true")
initiator("pkill -f [t]cpdump 2>/dev/null; sleep 1; true");

In [ ]:
# The ML-KEM pubkey always splits into exactly 2 fragments (Beat 2 measures the
# exact sizes); draw both arrows now, bolded, but no sizes yet: that's Beat 2's job
ipsec.stage.set_fragments("IKE_INTERMEDIATE  (ML-KEM pubkey)", ["", ""])
panel(ipsec.stage, step="IKE_AUTH  (confirm)", focus="IKE_INTERMEDIATE  (ML-KEM pubkey)", status=("ok", "ok"),
      rows=[("IKE SA suite", "CURVE_25519/KE1_ML_KEM_768", suite.text or "(no SA line)",
             "ML_KEM" in suite.text.upper())],
      notes=[("Round trips", "3 (adds the IKE_INTERMEDIATE exchange)")],
      banner="IKEv2 carries ML-KEM in an extra IKE_INTERMEDIATE round trip (RFC 9370) that classical IKEv2 does not have.")

**Beat 2: the fragmented ML-KEM key share on the wire**

In [ ]:
# Initiator: dump the raw capture; the fragmented IKE_INTERMEDIATE request is
# the biggest burst of back-to-back outbound packets (before the responder replies)
raw = initiator("tcpdump -r /tmp/hybrid.pcap -n -q 2>/dev/null")

In [ ]:
# Group the outbound packet sizes into bursts and keep the biggest one; that
# burst is the fragmented message; each element is one fragment's own size
bursts, cur = [], []
for line in raw.text.splitlines():
    if ' Out ' in line:
        cur.append(int(line.split()[-1]))
    elif ' In ' in line and cur:
        bursts.append(cur); cur = []
if cur:
    bursts.append(cur)
frags = [str(n) for n in max(bursts, key=sum)] if bursts else ['?']
total = str(sum(int(n) for n in frags))
print(f'fragments: {frags} (total {total} B)')

In [ ]:
# Draw one arrow per fragment actually seen on the wire, each tagged with its own size,
# this is what differentiates this panel from Beat 1's (same 2 arrows, now with real sizes)
last_step = ipsec.stage.set_fragments("IKE_INTERMEDIATE  (ML-KEM pubkey)", [f"{n} B" for n in frags])
panel(ipsec.stage, step=last_step, focus="IKE_INTERMEDIATE  (ML-KEM pubkey)", status=("ok", "ok"),
      rows=[("IKE_INTERMEDIATE message, before fragmentation", ">1280 B (~1322)", total + " B",
             total.isdigit() and int(total) > 1280),
            (f"Fragment 1/{len(frags)} on the wire", "~1252 B", frags[0] + " B",
             frags[0].isdigit() and abs(int(frags[0]) - 1252) <= 40),
            (f"Fragment {len(frags)}/{len(frags)} on the wire", "~70 B", frags[-1] + " B",
             frags[-1].isdigit() and abs(int(frags[-1]) - 70) <= 40)],
      banner=f"The ML-KEM public key makes the IKE_INTERMEDIATE message ~1322 B, past the ~1280 B IKE ceiling, so strongSwan splits it into {len(frags)} fragments.",
      banner_color=SLATE)

In [ ]:
# reset (terminate the tunnel, stop the capture)
initiator("swanctl --terminate --ike pqc-tunnel 2>/dev/null || true; true")
initiator("pkill -f [t]cpdump 2>/dev/null; true");

<a id="auth-ipsec"></a>
## IKEv2 Authentication

ML-KEM key exchange with **classical ECDSA authentication** first, and then the frontier/experimental **ML-DSA auth**,where big certs force `IKE_AUTH` to fragment.

**Beat 1: mutually authenticate with ECDSA certs**

In [ ]:
# host: mint ECDSA credentials for the two strongSwan peers (stack is
#       already up from preflight_up())
host("cd learn/ipsec/authentication && docker compose run --rm certgen ecdsa");

In [ ]:
# both peers: load the freshly minted credentials
auth_responder("swanctl --load-all")
auth_initiator("swanctl --load-all");

In [ ]:
# Initiator: bring up the tunnel; the log line names the authentication method
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true")
res = auth_initiator("swanctl --initiate --child auth-child")
suite = auth_initiator("swanctl --list-sas | grep -iE 'ML_KEM|CURVE_25519' || true")

In [ ]:
# Tag the IKE_AUTH arrows with what actually just ran, and bold them,
# this beat is about the authentication method, not the key exchange above it
ipsec_auth.stage.retag("IKE_AUTH  (cert + signature)", "ECDSA", direction="r")
panel(ipsec_auth.stage, step=5, focus="IKE_AUTH  (cert + signature)", status=("ok", "ok"),
      rows=[("Authentication method", "ECDSA_WITH_SHA256_DER",
             (grep(res.text, 'authentication of') or ['?'])[-1], 'ecdsa_with_sha256' in res.text.lower()),
            ("Key-exchange suite", "CURVE_25519/KE1_ML_KEM_768", suite.text, "ML_KEM" in suite.text.upper())],
      banner="Today's posture (e.g. 5G fronthaul): post-quantum ML-KEM key exchange with classical ECDSA authentication.",
      banner_color=BLUE)

In [ ]:
# reset the ECDSA tunnel before the ML-DSA rebuild
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true");

**Beat 2 (bleeding edge): ML-DSA-44 certificate authentication**

Switches the peers to the experimental strongSwan `master` image (already
compiled by `preflight_build()`) and reissues ML-DSA-44 certs.

In [ ]:
# host: switch to the experimental strongSwan master image, reissue ML-DSA-44 certs
host("cd learn/ipsec/authentication && docker compose -f docker-compose.yml -f docker-compose.mldsa.yml up -d")
host("cd learn/ipsec/authentication && docker compose -f docker-compose.yml -f docker-compose.mldsa.yml run --rm certgen ml-dsa-44");

In [ ]:
# both peers: load the ML-DSA credentials, then initiate
auth_responder("swanctl --load-all")
auth_initiator("swanctl --load-all")
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true")
res = auth_initiator("swanctl --initiate --child auth-child")

In [ ]:
# Retag the same arrows for this beat's actual auth method
ipsec_auth.stage.retag("IKE_AUTH  (cert + signature)", "ML-DSA-44", direction="r")
panel(ipsec_auth.stage, step=5, focus="IKE_AUTH  (cert + signature)", status=("ok", "ok"),
      rows=[("Authentication method", "ML_DSA_44",
             (grep(res.text, 'authentication of') or ['?'])[-1], 'ml_dsa_44' in res.text.lower())],
      banner="Frontier: both halves post-quantum (ML-KEM + ML-DSA). Experimental strongSwan master build; wire format still an IETF draft.",
      banner_color=MAGENTA)

**Beat 3: watch the ML-DSA cert + signature fragment IKE_AUTH**

In [ ]:
# host: search the initiator's log for IKE fragmentation lines, then pick out
# the biggest one; that's always the IKE_AUTH split (ML-KEM's is always ~1.2 KB)
frag = host("docker logs ike-auth-initiator 2>&1 | grep -i 'splitting IKE message'")
matches = re.findall(r'\((\d+) bytes\) into (\d+) fragments', frag.text)
auth_bytes, auth_frags = max(matches, key=lambda m: int(m[0])) if matches else ("6904", "6")
auth_frags = int(auth_frags)

In [ ]:
# Draw one arrow per real fragment: the fixed 2-way ML-KEM split (every run),
# plus one arrow per IKE_AUTH fragment, each tagged with its share of the real total
ipsec_auth.stage.set_fragments("IKE_INTERMEDIATE  (ML-KEM key share)", ["1252 B", "70 B"])
avg = int(auth_bytes) // auth_frags
last_step = ipsec_auth.stage.set_fragments("IKE_AUTH  (cert + signature)", [f"~{avg} B avg"] * auth_frags)

In [ ]:
panel(ipsec_auth.stage, step=last_step, focus="IKE_AUTH  (cert + signature)", status=("ok", "ok"),
      rows=[("IKE_INTERMEDIATE split (ML-KEM, every run)", "1249 B -> 2 fragments", "1249 B -> 2 fragments", True),
            ("IKE_AUTH split (ML-DSA cert + signature)", ">1280 B -> several fragments",
             f"{auth_bytes} B -> {auth_frags} fragments", int(auth_bytes) > 1280 and auth_frags > 2)],
      banner=f"The ML-DSA leaf cert + signature blow IKE_AUTH up to ~{auth_bytes} B -> {auth_frags} fragments; the ~1.2 KB ML-KEM split happens either way.",
      banner_color=SLATE)

In [ ]:
# reset (terminate the ML-DSA tunnel)
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true");

---
# TLS 1.3 (the web's secure channel)

<a id="kex-tls"></a>
## TLS 1.3 Key Exchange: hybrid key exchange, no extra round trip

The fast, safe default. One ClientHello / ServerHello still carries
ML-KEM; only the key-share **bytes** grow (1216 / 1120 vs. 32 / 32). No new
flight, no fragmentation.

**Beat 1: start the hybrid TLS 1.3 server + capture**

In [ ]:
# TLS server: mint a throwaway EC P-256 cert for the listener
tls_server("openssl genpkey -algorithm EC -pkeyopt ec_paramgen_curve:P-256 -out /tmp/server.key 2>/dev/null; echo key-ready")
tls_server("openssl req -x509 -new -key /tmp/server.key -out /tmp/server.crt -days 365 -subj /CN=tls-server.pqc.lab 2>/dev/null; echo cert-ready");

In [ ]:
# TLS server: packet capture, then a hybrid TLS 1.3 listener
tls_server("pkill -f [s]_server 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true")
tls_server("tcpdump -i any -U 'tcp port 4433' -w /tmp/hybrid.pcap &")
tls_server("openssl s_server -accept 4433 -cert /tmp/server.crt -key /tmp/server.key -tls1_3 -groups X25519MLKEM768 -www &");

**Beat 2: client connects, prints the negotiated group**

In [ ]:
# TLS client: connect and print the negotiated TLS 1.3 group
neg = tls_client("openssl s_client -connect 172.22.0.2:4433 -groups X25519MLKEM768 </dev/null 2>&1 | grep -iE 'Negotiated TLS1.3 group|Protocol|Cipher is'")

In [ ]:
panel(tls.stage, step=1, focus=["ClientHello", "ServerHello"], status=("active", "ok"),
      rows=[("Negotiated TLS 1.3 group", "X25519MLKEM768",
             (grep(neg.text, 'Negotiated') or ['?'])[0], 'MLKEM' in neg.text.upper())],
      banner="One round trip: the group is chosen in the ClientHello/ServerHello, same as classical.")

**Beat 3: read the ML-KEM key-share sizes off the capture, client vs. server**

In [ ]:
# TLS server: tcpdump needs a moment to drain its buffer before we stop it
tls_server("pkill -f [s]_server 2>/dev/null; sleep 2; pkill -f [t]cpdump 2>/dev/null; sleep 1; true");

In [ ]:
# TLS server: split the capture into the ClientHello vs. ServerHello key share
client_share = tls_server("tshark -r /tmp/hybrid.pcap -V 2>/dev/null | awk '/Handshake Type: Client Hello/{f=1} /Handshake Type: Server Hello/{f=0} f' | grep -i 'Key Share Entry: Group:' | head -1")
server_share = tls_server("tshark -r /tmp/hybrid.pcap -V 2>/dev/null | awk '/Handshake Type: Server Hello/{f=1} f' | grep -i 'Key Share Entry: Group:' | head -1")

In [ ]:
panel(tls.stage, step=3, focus=["ClientHello", "ServerHello"], status=("ok", "ok"),
      rows=[("Client -> Server key share (in ClientHello)", "1216 B", kex_len(client_share.text) + " B",
             kex_len(client_share.text) == "1216"),
            ("Server -> Client key share (in ServerHello)", "1120 B", kex_len(server_share.text) + " B",
             kex_len(server_share.text) == "1120")],
      notes=[("Round trips", "1 (no extra flight)")],
      banner="Hybrid key exchange with no extra round trip: the cost is just bytes, not a new flight.")

**Beat 4 (optional): classical x25519 for contrast (32 B shares, still 1-RTT)**

In [ ]:
# TLS server: restart the listener classical-only (x25519)
tls_server("pkill -f [s]_server 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true")
tls_server("tcpdump -i any -U 'tcp port 4433' -w /tmp/classical.pcap &")
tls_server("openssl s_server -accept 4433 -cert /tmp/server.crt -key /tmp/server.key -tls1_3 -groups x25519 -www &");

In [ ]:
# TLS client: connect classical-only
tls_client("openssl s_client -connect 172.22.0.2:4433 -groups x25519 </dev/null 2>&1 | grep -iE 'Negotiated TLS1.3 group'");

In [ ]:
# TLS server: drain, stop, and read the classical key-share size
tls_server("pkill -f [s]_server 2>/dev/null; sleep 2; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
classical_share = tls_server("tshark -r /tmp/classical.pcap -V 2>/dev/null | grep -i 'Key Share Entry: Group:' | head -1")

In [ ]:
# Retag the same two arrows for this contrast run; they said '...hybrid share'
# a moment ago; x25519-only shares are the same 32 B on both sides
classical_tag = kex_len(classical_share.text) + ' B classical share'
tls.stage.retag("ClientHello", classical_tag, direction="r")
tls.stage.retag("ServerHello", classical_tag, direction="l")
panel(tls.stage, step=1, focus=["ClientHello", "ServerHello"], status=("ok", "ok"),
      rows=[("Classical key share (each side)", "32 B", kex_len(classical_share.text) + " B",
             kex_len(classical_share.text) == "32")],
      banner="Classical x25519: 32-byte shares, same single round trip. The hybrid cost is purely bytes.",
      banner_color=SLATE)

In [ ]:
# reset (stop the TLS listener + capture)
tls_server("pkill -f [s]_server 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true");

<a id="auth-tls"></a>
## TLS 1.3 Authentication: mutual authentication (`tls/authentication`)

Key exchange was the fire alarm; **authentication is the slow
renovation**. The hybrid ML-KEM key exchange stays on; only the certificate
**signature** swaps ECDSA -> ML-DSA. On TCP that just makes the handshake
longer (no fragmentation, unlike IKEv2).

**Beat 1: weigh the certificates (ML-DSA-65 is ~15x the ECDSA leaf)**

In [ ]:
# TLS server: mint one cert per algorithm and list DER sizes
mtls_server("cd /tmp && for e in EC:ecdsa ML-DSA-44:mldsa44 ML-DSA-65:mldsa65 ML-DSA-87:mldsa87; do a=${e%%:*}; n=${e##*:}; if [ $a = EC ]; then openssl genpkey -algorithm EC -pkeyopt ec_paramgen_curve:P-256 -out $n.key 2>/dev/null; else openssl genpkey -algorithm $a -out $n.key 2>/dev/null; fi; openssl req -x509 -new -key $n.key -out $n.crt -days 365 -subj /CN=$n -outform DER 2>/dev/null; done; ls -l *.crt | awk '{print $5, $9}' | sort -n");

**Beat 2: mutual TLS with classical ECDSA certificates**

In [ ]:
# host: mint an ECDSA CA + server/client certs
host("cd learn/tls/authentication && docker compose run --rm certgen ecdsa");

In [ ]:
# TLS server: require + verify a client certificate (mutual TLS)
mtls_server("pkill -f [s]_server 2>/dev/null; true")
mtls_server("openssl s_server -accept 4433 -cert /cfg/server.crt -key /cfg/server.key -CAfile /cfg/ca.crt -Verify 1 -tls1_3 -www &");

In [ ]:
# TLS client: present its own certificate and connect
res = mtls_client("openssl s_client -connect 172.23.0.2:4433 -cert /cfg/client.crt -key /cfg/client.key -CAfile /cfg/ca.crt </dev/null 2>&1 | grep -iE 'Verify return code|Peer signature type|SSL handshake has read'")
mtls_server("pkill -f [s]_server 2>/dev/null; true")
reads = first_int((grep(res.text, 'has read') or [''])[0])

In [ ]:
panel(tls_auth.stage, step=3,
      focus=["ServerHello + Certificate", "Client Certificate"], status=("ok", "ok"),
      rows=[("Peer signature type", "ecdsa_secp256r1_sha256", (grep(res.text, 'signature type') or ['?'])[0], 'ecdsa' in res.text.lower()),
            ("Verify return code", "0 (ok)", (grep(res.text, 'Verify return') or ['?'])[0], '0 (ok)' in res.text)],
      notes=[("Handshake bytes read", reads + ' B (~2.5 KB)')],
      banner="Baseline: mutual TLS with classical ECDSA certs over a hybrid ML-KEM key exchange.")

**Beat 3: same handshake with ML-DSA-65 certificates**

In [ ]:
# host: reissue the CA + certs as ML-DSA-65
host("cd learn/tls/authentication && docker compose run --rm certgen ml-dsa-65");

In [ ]:
# TLS server + client: same mutual-TLS handshake, new certificates
mtls_server("pkill -f [s]_server 2>/dev/null; true")
mtls_server("openssl s_server -accept 4433 -cert /cfg/server.crt -key /cfg/server.key -CAfile /cfg/ca.crt -Verify 1 -tls1_3 -www &")
res = mtls_client("openssl s_client -connect 172.23.0.2:4433 -cert /cfg/client.crt -key /cfg/client.key -CAfile /cfg/ca.crt </dev/null 2>&1 | grep -iE 'Verify return code|Peer signature type|SSL handshake has read'")
mtls_server("pkill -f [s]_server 2>/dev/null; true")
reads = first_int((grep(res.text, 'has read') or [''])[0])

In [ ]:
panel(tls_auth.stage, step=3,
      focus=["ServerHello + Certificate", "Client Certificate"], status=("ok", "ok"),
      rows=[("Peer signature type", "mldsa65", (grep(res.text, 'signature type') or ['?'])[0], 'mldsa65' in res.text.lower()),
            ("Verify return code", "0 (ok)", (grep(res.text, 'Verify return') or ['?'])[0], '0 (ok)' in res.text)],
      notes=[("Handshake bytes read", reads + ' B (~16 KB)')],
      banner="Same handshake, ML-DSA-65 certs: ~2.5 KB -> ~16 KB. Both halves now quantum-safe (ML-KEM + ML-DSA).",
      banner_color=MAGENTA)

---
# MACsec / 802.1X (Layer 2 link encryption)

<a id="macsec"></a>
## MACsec / EAP-TLS Key Exchange & Authentication: the certificate becomes visible frames

The hybrid ML-KEM key exchange is **negotiated inside the EAP-TLS
handshake**. When we swap a small ECDSA cert for a large ML-DSA cert, the key
exchange is unchanged; only the **certificate** grows, and that multiplies
**EAPOL frames** and fragments. That is the operational tax.

**Beat 1: baseline handshake with a classical ECDSA certificate**

In [ ]:
# host: mint an ECDSA CA + server/client certs into the lab config dirs
host("cd learn/macsec && docker compose run --rm certgen ecdsa");

In [ ]:
# Authenticator: capture EAPOL (ether proto 0x888e), then start the hostapd EAP server
authenticator("pkill -f [h]ostapd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true")
authenticator("tcpdump -i aut -U 'ether proto 0x888e' -w /tmp/ecdsa.pcap &")
authenticator("hostapd -dd /cfg/hostapd.conf > /tmp/h.log 2>&1 &");

In [ ]:
# Supplicant: run one EAP-TLS handshake against the authenticator
supplicant("pkill wpa_supplicant 2>/dev/null; rm -f /run/wpa/sup; true")
supplicant("wpa_supplicant -Dwired -isup -c /cfg/wpa_supplicant.conf -t -dd > /tmp/w.log 2>&1 &")
supplicant("sleep 6; grep -aoE 'CTRL-EVENT-EAP-SUCCESS' /tmp/w.log | head -1");

In [ ]:
# Authenticator: stop the capture, count EAPOL frames, prove the hybrid KE group
authenticator("pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
frames = authenticator("tshark -r /tmp/ecdsa.pcap 2>/dev/null | wc -l")
group  = authenticator("tshark -r /tmp/ecdsa.pcap -V 2>/dev/null | awk '/Handshake Type: Server Hello/{f=1} f' | grep -i 'Key Share Entry: Group:' | head -1")
authenticator("pkill -f [h]ostapd 2>/dev/null; true")
supplicant("pkill wpa_supplicant 2>/dev/null; true");

In [ ]:
panel(macsec.stage, step=6,
      focus=["EAP-TLS ClientHello", "ServerHello + Certificate"], status=("ok", "ok"),
      rows=[("EAPOL frames (ECDSA leaf cert)", "14", frames.text,
             frames.text.isdigit() and abs(int(frames.text) - 14) <= 2),
            ("ServerHello key-exchange group", "X25519MLKEM768", group.text or "(not found)",
             "MLKEM" in group.text.upper())],
      banner="The hybrid ML-KEM key exchange is negotiated inside this EAP-TLS handshake; here the certificate is classical ECDSA.")

**Beat 2: reissue the SAME identity as ML-DSA-65 (only the cert changes)**

In [ ]:
# host: reissue the same identity's certificate as ML-DSA-65
host("cd learn/macsec && docker compose run --rm certgen ml-dsa-65");

In [ ]:
# Authenticator: fresh capture, EAP server with the new certificate
authenticator("pkill -f [h]ostapd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
authenticator("tcpdump -i aut -U 'ether proto 0x888e' -w /tmp/mldsa.pcap &")
authenticator("hostapd -dd /cfg/hostapd.conf > /tmp/h.log 2>&1 &");

In [ ]:
# Supplicant: same identity, one more handshake
supplicant("pkill wpa_supplicant 2>/dev/null; rm -f /run/wpa/sup; true")
supplicant("wpa_supplicant -Dwired -isup -c /cfg/wpa_supplicant.conf -t -dd > /tmp/w.log 2>&1 &")
supplicant("sleep 6; grep -aoE 'CTRL-EVENT-EAP-SUCCESS' /tmp/w.log | head -1");

In [ ]:
# Authenticator: stop the capture, count EAPOL frames again
authenticator("pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
frames = authenticator("tshark -r /tmp/mldsa.pcap 2>/dev/null | wc -l")
authenticator("pkill -f [h]ostapd 2>/dev/null; true")
supplicant("pkill wpa_supplicant 2>/dev/null; true");

In [ ]:
panel(macsec.stage, step=6, focus="ServerHello + Certificate", status=("ok", "ok"),
      rows=[("EAPOL frames (ML-DSA-65 leaf cert)", "54", frames.text,
             frames.text.isdigit() and abs(int(frames.text) - 54) <= 4)],
      notes=[("Only change from the baseline", "ECDSA cert -> ML-DSA-65 cert (key exchange untouched)")],
      banner="Same handshake, one variable changed (the certificate) and EAPOL frames jump 14 -> 54.",
      banner_color=MAGENTA)

**Beat 3 (optional): the SILENT downgrade: drop the TLS 1.3 gate**

In [ ]:
# TLS 1.2 cannot use an ML-DSA cert, so put a classical one back first
host("cd learn/macsec && docker compose run --rm certgen ecdsa")
authenticator("sed -i 's/^tls_flags=/#tls_flags=/' /cfg/hostapd.conf")
authenticator("grep -n tls_flags /cfg/hostapd.conf");

In [ ]:
# Authenticator: fresh capture, EAP server with the gate open
authenticator("pkill -f [h]ostapd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
authenticator("tcpdump -i aut -U 'ether proto 0x888e' -w /tmp/down.pcap &")
authenticator("hostapd -dd /cfg/hostapd.conf > /tmp/h.log 2>&1 &");

In [ ]:
# Supplicant: one more handshake
supplicant("pkill wpa_supplicant 2>/dev/null; rm -f /run/wpa/sup; true")
supplicant("wpa_supplicant -Dwired -isup -c /cfg/wpa_supplicant.conf -t -dd > /tmp/w.log 2>&1 &")
supplicant("sleep 6; grep -aoE 'CTRL-EVENT-EAP-SUCCESS' /tmp/w.log | head -1");

In [ ]:
# Authenticator: stop the capture, read the negotiated cipher off the ServerHello
authenticator("pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
cipher = authenticator("tshark -r /tmp/down.pcap -V 2>/dev/null | awk '/Handshake Protocol: Server Hello/{f=1} f' | grep -iE 'Cipher Suite: TLS_ECDHE' | head -1")

In [ ]:
# Restore the TLS 1.3 gate and stop the daemons
authenticator("sed -i 's/^#tls_flags=/tls_flags=/' /cfg/hostapd.conf")
authenticator("grep -n tls_flags /cfg/hostapd.conf")
authenticator("pkill -f [h]ostapd 2>/dev/null; true")
supplicant("pkill wpa_supplicant 2>/dev/null; true");

In [ ]:
panel(macsec.stage, step=6, focus="ServerHello + Certificate", status=("warn", "warn"),
      rows=[("Fell back to classical TLS 1.2", "TLS_ECDHE_ECDSA...", cipher.text or "(none)",
             "ecdhe" in cipher.text.lower())],
      banner="MACsec fallback is SILENT: EAP still succeeds, but the ServerHello shows classical TLS 1.2 with no ML-KEM. Contrast the loud SSH warning below.",
      banner_color=ORANGE)

In [ ]:
# reset (stop hostapd/wpa_supplicant + capture)
authenticator("pkill -f [h]ostapd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true")
supplicant("pkill wpa_supplicant 2>/dev/null; true");

---
# SSH (secure remote access)

<a id="ssh"></a>
## SSH Key Exchange & Authentication: post-quantum key exchange, on by default

Modern OpenSSH negotiates hybrid ML-KEM with **zero config**, and a
classical-only server triggers a **loud client warning** (contrast MACsec's
silent fallback above). Composite ML-DSA host/user keys make **both** pillars PQ.

**Beat 1: start sshd, connect with no KEX config**

In [ ]:
# SSH server: start sshd with the shipped config (hybrid KEX by default)
ssh_server("pkill -f [s]shd 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config > /tmp/sshd.log 2>&1 &");

In [ ]:
# SSH client: zero KEX config; show the negotiated algorithm
kex = ssh_client("ssh -F /cfg/ssh_config -v ssh-server 'echo AUTH_OK' 2>&1 | grep -E 'kex: algorithm|Authenticated to|AUTH_OK'")

In [ ]:
panel(ssh.stage, step=4, focus=["KEX init", "KEX reply"], status=("ok", "ok"),
      rows=[("Key-exchange algorithm", "mlkem768x25519-sha256",
             (grep(kex.text, 'kex: algorithm') or ['?'])[0], 'mlkem768x25519' in kex.text)],
      notes=[("Client / server KEX bytes", "1304 B / 1716 B"), ("Round trips", "1 (same as classical)")],
      banner="Modern OpenSSH negotiates hybrid ML-KEM with zero configuration.")

**Beat 2: model an un-upgraded server (classical-only KEX): LOUD warning**

In [ ]:
# SSH server: restart offering only a classical KEX
ssh_server("pkill -f [s]shd 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config -o KexAlgorithms=curve25519-sha256 > /tmp/sshd.log 2>&1 &");

In [ ]:
# SSH client still offers hybrid, is forced down, and warns
warn = ssh_client("ssh -F /cfg/ssh_config ssh-server 'true' 2>&1 | grep -iE 'WARNING|post-quantum|store now' || true")

In [ ]:
# The KEX init/reply arrows below still say '1304 B'/'1716 B' from Beat 1; retag
# them for this downgraded run, and bold the KEXINIT negotiation where it happens
ssh.stage.retag("KEX init", "x25519 only, no ML-KEM", direction="r")
ssh.stage.retag("KEX reply", "x25519 only, no ML-KEM", direction="l")
panel(ssh.stage, step=1, focus="KEXINIT", status=("warn", "warn"),
      rows=[("Client downgrade warning", "** WARNING ...", warn.text or "(none)", bool(warn.text.strip()))],
      banner="SSH downgrade is LOUD: the client warns the session is not post-quantum. Contrast the silent MACsec TLS 1.2 fallback above.",
      banner_color=ORANGE)

**Beat 3: restore the PQ-capable default**

In [ ]:
# SSH server: restart with the shipped (hybrid) config
ssh_server("pkill -f [s]shd 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config > /tmp/sshd.log 2>&1 &")
print('sshd restored to the hybrid default')
# Restore the KEX arrow tags now that the server is hybrid again
ssh.stage.retag("KEX init", "1304 B", direction="r")
ssh.stage.retag("KEX reply", "1716 B", direction="l")

**Beat 4 (optional): composite ML-DSA-44 + Ed25519 authentication (both pillars PQ)**

In [ ]:
# host: reissue host + user keys as composite mldsa44-ed25519
host("cd learn/ssh && docker compose run --rm keygen mldsa44-ed25519");

In [ ]:
# SSH server: offer the composite host key, start a capture
ssh_server("pkill -f [s]shd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config -o HostKeyAlgorithms=ssh-mldsa44-ed25519@openssh.com -o PubkeyAcceptedAlgorithms=ssh-mldsa44-ed25519@openssh.com > /tmp/sshd.log 2>&1 &")
ssh_server("tcpdump -i any -U 'tcp port 22' -w /tmp/comp.pcap &");

In [ ]:
# SSH client: authenticate with the composite type
auth = ssh_client("ssh -F /cfg/ssh_config -o HostKeyAlgorithms=ssh-mldsa44-ed25519@openssh.com -o PubkeyAcceptedAlgorithms=ssh-mldsa44-ed25519@openssh.com -v ssh-server 'echo PQ_AUTH_OK' 2>&1 | grep -iE 'host key algorithm|Server accepts key|PQ_AUTH_OK'")

In [ ]:
# SSH server: tcpdump needs a moment to drain, then read the cleartext sizes
ssh_server("sleep 2; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
sizes = ssh_server("tshark -r /tmp/comp.pcap -V 2>/dev/null | grep -iE 'Host key length|Host signature length'")
hostkey = first_int((grep(sizes.text, 'Host key length') or [''])[0])
hostsig = first_int((grep(sizes.text, 'Host signature length') or [''])[0])

In [ ]:
# The 'KEX reply' arrow is where the host key + its signature actually ride,
# retag it with what we just measured, and bold it (not the auth arrow after it)
ssh.stage.retag("KEX reply", f"host key {hostkey} B + sig {hostsig} B", direction="l")
panel(ssh.stage, step=4, focus="KEX reply", status=("ok", "ok"),
      rows=[("Composite host key", "1383 B", hostkey + ' B', hostkey == '1383'),
            ("Composite host signature", "2523 B", hostsig + ' B', hostsig == '2523')],
      banner="Both pillars post-quantum: hybrid ML-KEM key exchange + composite ML-DSA-44 authentication (opt-in).",
      banner_color=MAGENTA)

In [ ]:
# reset (stop sshd + capture)
ssh_server("pkill -f [s]shd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true");

---
<a id="wrapup"></a>
## Cleanup

`panic_reset()` clears stray processes; `cleanup_all()` restores shipped defaults
(containers down, lab configs/creds reset).

In [ ]:
panic_reset()

In [ ]:
cleanup_all()